# Processamento Inicial

Nesse notebook faremos um processamento inicial do dataset bruto, convertendo os tipos das features adequadamente para inteiros, floats, tipos categóricos e datas. Além disso faremos uma etapa de sanitização para converter dados inconsistentes em ´NA´ e salvando o dataframe resultante em formato parquet.

## Importando as Bibliotecas

Primeiro vamos importar as bibliotecas que usaremos.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Carregando o Dataset

Agora vamos carregar o dataset.

In [2]:
df = pd.read_csv("../data/raw/UCMF_raw.csv")

## Análise Inicial

Vamos fazer um análise inicial, olhando para um trecho do `DataFrame` e analisando as informações mais básicas dele.

In [3]:
df.head()

,ID,Peso,Altura,IMC,Atendimento,DN,IDADE,Convenio,PULSOS,PA SISTOLICA,...,PPA,NORMAL X ANORMAL,B2,SOPRO,FC,HDA 1,HDA2,SEXO,MOTIVO1,MOTIVO2
0,1,5.0,51,19.0,11/05/06,30/03/06,0.12,GS,Normais,NaN,...,Não Calculado,Anormal,Normal,Sistólico,112,Palpitacao,NaN,M,6 - Suspeita de cardiopatia,6 - Palpitação/taquicardia/arritmia
1,2,3.5,50,14.0,25/05/05,19/05/05,0.02,GS,Normais,NaN,...,Não Calculado,Anormal,Normal,ausente,128,Dispneia,NaN,M,6 - Suspeita de cardiopatia,6 - Dispnéia
2,3,0.0,0,NaN,12/06/01,08/05/05,-4.05,SULA,Normais,NaN,...,Não Calculado,Anormal,Normal,Sistólico,88,Assintomático,NaN,M,2 - Check-up,NaN
3,4,8.1,65,19.0,15/10/09,21/04/09,0.50,NaN,Normais,NaN,...,Não Calculado,Anormal,Normal,ausente,92,Assintomático,NaN,M,5 - Parecer cardiológico,NaN
4,7,40.0,151,18.0,14/01/08,14/08/95,12.89,SAME,Normais,NaN,...,Não Calculado,Anormal,Normal,ausente,96,Dor precordial,NaN,M,5 - Parecer cardiológico,NaN


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                12873 non-null  int64  
 1   Peso              12555 non-null  float64
 2   Altura            12873 non-null  int64  
 3   IMC               8146 non-null   float64
 4   Atendimento       11890 non-null  str    
 5   DN                11497 non-null  str    
 6   IDADE             11377 non-null  float64
 7   Convenio          8765 non-null   str    
 8   PULSOS            11679 non-null  str    
 9   PA SISTOLICA      5143 non-null   float64
 10  PA DIASTOLICA     5133 non-null   float64
 11  PPA               12656 non-null  str    
 12  NORMAL X ANORMAL  11705 non-null  str    
 13  B2                11695 non-null  str    
 14  SOPRO             11706 non-null  str    
 15  FC                11005 non-null  str    
 16  HDA 1             8603 non-null   str    
 17  HDA2

Nosso dataset tem, inicialmente, $21$ features e $12873$ registros, sendo a feature `"NORMAL X ANORMAL"` o nosso target.

Como podemos ver alguns nomes de features estão em maiúsculo e com espaços. Vamos então limpar esses nomes e adotar a seguinte convenção: nomes com letras minúsculas e underscore (`_`) como separador. Além disso vamos renomear as features `"DN"`, `"HDA 1"` e `"NORMAL X ANORMAL"` em `"nascimento"`, `"hda1"` e `"patologia"`, respectivamente.

In [5]:
def normalize_names(x):
    return "_".join(x.strip().lower().split())

df = (df
    .rename(lambda x: normalize_names(x), axis="columns")
    .rename({
        "dn": "nascimento",
        "hda_1": "hda1",
        "normal_x_anormal": "patologia",
    }, axis="columns")
)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             12873 non-null  int64  
 1   peso           12555 non-null  float64
 2   altura         12873 non-null  int64  
 3   imc            8146 non-null   float64
 4   atendimento    11890 non-null  str    
 5   nascimento     11497 non-null  str    
 6   idade          11377 non-null  float64
 7   convenio       8765 non-null   str    
 8   pulsos         11679 non-null  str    
 9   pa_sistolica   5143 non-null   float64
 10  pa_diastolica  5133 non-null   float64
 11  ppa            12656 non-null  str    
 12  patologia      11705 non-null  str    
 13  b2             11695 non-null  str    
 14  sopro          11706 non-null  str    
 15  fc             11005 non-null  str    
 16  hda1           8603 non-null   str    
 17  hda2           407 non-null    str    
 18  sexo           12

Agora temos nomes mais apropriados para as features e seguindo uma convenção adequada. Vamos então analisar feature por feature para convertê-las em tipos mais apropriadas e garantir a consistência dos dados.

## Features Temporais (Datas)

Agora, vamos processar as features temporais `"atendimento"` e `"nascimento"`. Ambas as features são do tipo string (`str`), portanto precisamos convertê-las para o tipo data (`datetime`). Vamos dar uma olhadinha nos formatos de datas que temos.

In [7]:
cols = ["atendimento", "nascimento"]

df[cols].head()

,atendimento,nascimento
0,11/05/06,30/03/06
1,25/05/05,19/05/05
2,12/06/01,08/05/05
3,15/10/09,21/04/09
4,14/01/08,14/08/95


Analisando esse trecho do `DataFrame`, podemos inferir que os dados estão no formato `"%d/%m/%y"`. Para verificar isso vamos ver quantos registros respeitam o padrão `\d{2}/\d{2}/\d{2}` (dois digitos para o dia, mês e ano) de regex no Python.

In [8]:
for col in cols:
    mask = df[col].str.match(r"^\d{2}/\d{2}/\d{2}$")
    mached = df[mask][col]
    count = mached.shape[0]

    print(f"{col}: {count}")

atendimento: 11578
nascimento: 11259


Como podemos ver, a maioria das datas estão realmente no padrão de regex proposto e por isso vamos converter essas features para `datetime`. Note que usamos o argumento `errors="coerce"` para que o pandas salve as datas que não puderam ser convertidas como `pd.NaT`.

In [9]:
format = "%d/%m/%y"

for col in cols:
    df[col] = pd.to_datetime(df[col], format=format, errors="coerce")

Agora, vamos checar se nossa conversão foi bem sucedida.

In [10]:
df[cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   atendimento  11578 non-null  datetime64[us]
 1   nascimento   11253 non-null  datetime64[us]
dtypes: datetime64[us](2)
memory usage: 201.3 KB


Como podemos notar, temos $11578$ e $11253$ datas convertidas nas features `"atendimento"` e `"nascimento"`, respectivamente. Então, vamos checar algumas estatísticas dessas datas.

In [11]:
df[cols].describe()

,atendimento,nascimento
count,11578,11253
mean,2007-03-11 06:01:40.742788,2002-02-26 14:47:18.922953
min,1990-01-03 00:00:00,1969-07-14 00:00:00
25%,2005-07-07 00:00:00,1998-08-15 00:00:00
50%,2007-05-30 00:00:00,2003-01-24 00:00:00
75%,2009-01-05 00:00:00,2006-03-27 00:00:00
max,2066-08-22 00:00:00,2068-07-06 00:00:00


Note como a maioria das datas estão entre os anos $1990$ e $2009$, mas algumas datas são de anos acima de $2026$, o que na prática é impossível já que essas datas são de events que já ocorreram. Vamos investigar isso.

In [12]:
nascimentos = df["nascimento"].copy()
atendimentos = df["atendimento"].copy()

nascimentos = (nascimentos.dt.year
    .sort_values()
    .value_counts(normalize=True, sort=False)
)

atendimentos = (atendimentos.dt.year
    .sort_values()
    .value_counts(normalize=True, sort=False)
)

print(nascimentos)
print(atendimentos)

nascimento
1969.0    0.000178
1970.0    0.000089
1971.0    0.000089
1972.0    0.000089
1973.0    0.000089
1974.0    0.000089
1975.0    0.000178
1976.0    0.000089
1977.0    0.000089
1978.0    0.000355
1979.0    0.000267
1980.0    0.000355
1981.0    0.000622
1982.0    0.000355
1983.0    0.000800
1984.0    0.000533
1985.0    0.000800
1986.0    0.001066
1987.0    0.002222
1988.0    0.003466
1989.0    0.005687
1990.0    0.008620
1991.0    0.011019
1992.0    0.017151
1993.0    0.019728
1994.0    0.031014
1995.0    0.033680
1996.0    0.036968
1997.0    0.042566
1998.0    0.048432
1999.0    0.049853
2000.0    0.054030
2001.0    0.061584
2002.0    0.062917
2003.0    0.082645
2004.0    0.077046
2005.0    0.075713
2006.0    0.079445
2007.0    0.069759
2008.0    0.067626
2009.0    0.047809
2010.0    0.003821
2022.0    0.000089
2027.0    0.000089
2036.0    0.000089
2040.0    0.000089
2051.0    0.000089
2055.0    0.000178
2064.0    0.000089
2065.0    0.000089
2068.0    0.000267
Name: proportion, dt

Como podemos observar, as datas estão, em sua maioria entre $1990$ e $2010$. Portanto vamos manter as datas nesse intervalo e as datas que estiverem fora nós vamos converter para `pd.NaT`.

In [13]:
for col in cols:
    df[col] = df[col].where(df[col].dt.year <= 2023)

In [14]:
df[cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   atendimento  11577 non-null  datetime64[us]
 1   nascimento   11242 non-null  datetime64[us]
dtypes: datetime64[us](2)
memory usage: 201.3 KB


Agora podemos olhar as infos do nosso dataset e ver que as features temporais já estão com seus tipos convertidos.

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             12873 non-null  int64         
 1   peso           12555 non-null  float64       
 2   altura         12873 non-null  int64         
 3   imc            8146 non-null   float64       
 4   atendimento    11577 non-null  datetime64[us]
 5   nascimento     11242 non-null  datetime64[us]
 6   idade          11377 non-null  float64       
 7   convenio       8765 non-null   str           
 8   pulsos         11679 non-null  str           
 9   pa_sistolica   5143 non-null   float64       
 10  pa_diastolica  5133 non-null   float64       
 11  ppa            12656 non-null  str           
 12  patologia      11705 non-null  str           
 13  b2             11695 non-null  str           
 14  sopro          11706 non-null  str           
 15  fc             11005 non-null 

## Features Numéricas

Vamos tratar as features numéricas. Para isso vamos olhar para as features `"peso"`, `"altura"`, `"idade"` e `"imc"` e mais tarde trataremos das features `"pa_sistolica"` e `"pa_diastolica"`.

### Features de Peso, Altura e Idade

Primeiro vamos olhar para algumas estatísticas dos dados de `"peso"`, `"altura"` e `"idade"`.

In [16]:
cols = ["peso", "altura", "idade"]

df[cols].describe()

,peso,altura,idade
count,12555.000000,12873.000000,11377.000000
mean,16.614417,66.111474,4.856947
std,16.979814,56.027934,8.829742
min,-40.000000,0.000000,-113.180000
25%,3.400000,0.000000,0.930000
50%,12.800000,69.000000,3.960000
75%,24.000000,112.000000,8.590000
max,157.000000,198.000000,71.810000


Como podemos ver, temos alguns registros negativos em `"peso"` e `"idade"`. Portanto vamos converter os dados negativos para `np.nan` e garantir que as $3$ features tenham tipo `np.float64`.

In [17]:
for col in cols:
    df[col] = df[col].astype(np.float64)

    if col == "idade":
        df[col] = df[col].where(df[col] >= 0.0)
    else:
        df[col] = df[col].where(df[col] > 0.0)

df[cols].describe()

,peso,altura,idade
count,9945.000000,8413.000000,11255.000000
mean,20.978783,101.159277,5.374211
std,16.498236,35.463332,5.129182
min,0.300000,10.000000,0.000000
25%,9.100000,71.000000,1.025000
50%,16.000000,100.000000,4.040000
75%,28.000000,129.000000,8.630000
max,157.000000,198.000000,71.810000


Podemos ver que os valores mínimos e máximos são consistentes com o que esperamos nesse dataset, o peso médio é de $20\text{kg}$, a altura média é de $1\text{m}$ e a idade média é de $5$ anos. Além disso os mínimos são todos não negativos e os valores máximos, apesar de serem um pouco extremos são valores possíveis.

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             12873 non-null  int64         
 1   peso           9945 non-null   float64       
 2   altura         8413 non-null   float64       
 3   imc            8146 non-null   float64       
 4   atendimento    11577 non-null  datetime64[us]
 5   nascimento     11242 non-null  datetime64[us]
 6   idade          11255 non-null  float64       
 7   convenio       8765 non-null   str           
 8   pulsos         11679 non-null  str           
 9   pa_sistolica   5143 non-null   float64       
 10  pa_diastolica  5133 non-null   float64       
 11  ppa            12656 non-null  str           
 12  patologia      11705 non-null  str           
 13  b2             11695 non-null  str           
 14  sopro          11706 non-null  str           
 15  fc             11005 non-null 

Por fim, podemos notar que `"peso"` está em quilos ($\text{kg}$), `"altura"` está em centimetros ($\text{cm}$) e `"idade"` está em anos.

### Feature de IMC

Agora vamos analisar a feature `"imc"`. Primeiro vamos olhar para algumas estatísticas dessa variável.

In [19]:
df[["imc"]].describe()

,imc
count,8146.000000
mean,17.455315
std,15.008016
min,0.000000
25%,15.000000
50%,17.000000
75%,19.000000
max,848.000000


Como podemos ver, existem alguns registros com $\text{IMC}$ iguais a $0$, então vamos checar quantos registros possuem o $\text{IMC}$ anulado.

In [20]:
(df["imc"] <= 1e-6).sum()

np.int64(109)

Temos poucos registros com $\text{IMC}$ nulo, então vamos transformar esses valores em `nan`.

In [21]:
df["imc"] = df["imc"].where(df["imc"] > 0.0)

In [22]:
df[["imc"]].describe()

,imc
count,8037.000000
mean,17.692049
std,14.970199
min,2.000000
25%,15.000000
50%,17.000000
75%,19.000000
max,848.000000


Agora, para finalizar com essa feature, vamos checar a consistência dela com as features `"peso"` e `"altura"`.

In [23]:
imc = df["peso"] / (df["altura"] / 100.0)**2
diff = imc - df["imc"]

print(f"quantidade de registros com IMC: {df["imc"].notna().sum()}")
print(f"quantidade de registros com IMC válidos: {(diff.abs() <= 1.0).sum()}")

quantidade de registros com IMC: 8037
quantidade de registros com IMC válidos: 8033


Como podemos ver, a maioria dos valores registrados de $\text{IMC}$ estão a uma unidade de diferença em relação ao valor do imc calculado a partir das features `"peso"` e `"altura"`. Portanto os valores de $\text{IMC}$ parecem estar consistentes com as duas features mencionadas. Dessa forma temos duas opções, manter esses valores ou recalculâ-los. Por enquanto, vamos manter os dados originais.

### Features de Pressão Arterial Sistólica e Diastólica

Agora, vamos tratar das features `"pa_sistolica"` e `"pa_diastolica"`. Vamos olhar algumas estatísticas sobre elas.

In [24]:
df[["pa_sistolica", "pa_diastolica"]].describe()

,pa_sistolica,pa_diastolica
count,5143.000000,5133.000000
mean,101.798561,62.646990
std,18.901140,9.092261
min,60.000000,6.000000
25%,90.000000,60.000000
50%,100.000000,60.000000
75%,110.000000,70.000000
max,990.000000,120.000000


Como podemos ver, menos da metade dos registros possuem valores para essas features, o que pode ser problemático na etapa de modelagem. Além disso o valor máximo da PA sistólica é $990$ e o mínimo da diastólica é $6$. Vamos checar os percentis de $0\%$ a $5\%$ e de $95\%$ a $100\%$ desses dados para checar esses valores extremos.

In [25]:
sistolica = df["pa_sistolica"]
diastolica = df["pa_diastolica"]

sistolica = sistolica[sistolica.notna()].sort_values()
diastolica = diastolica[diastolica.notna()].sort_values()

quantiles = np.linspace(0, 1, 101)

q_df = pd.DataFrame({
    "quantil": (100*quantiles).round(),
    "sistolica": np.quantile(sistolica, quantiles),
    "diastolica": np.quantile(diastolica, quantiles)
})

q_edge_df = pd.concat([q_df.iloc[:5], q_df.iloc[95:]], axis="index")
q_edge_df

,quantil,sistolica,diastolica
0,0.0,60.0,6.0
1,1.0,80.0,40.0
2,2.0,80.0,50.0
3,3.0,85.0,50.0
4,4.0,90.0,50.0
95,95.0,120.0,80.0
96,96.0,120.0,80.0
97,97.0,130.0,80.0
98,98.0,130.0,90.0
99,99.0,140.0,90.0


Como podemos ver, esses valores extremos parecem ser apenas outliers e como essas variáveis são mais sensíveis, vamos manter seus dados inalterados por enquanto.

## Features Categóricas

Agora vamos tratar das features categóricas. Antes disso, vamos checar quais features são realmente categóricas.

In [26]:
df_str = df.select_dtypes(include="str")

def show_categories(df_c):
    print(f"features: {df_c.columns.tolist()}\n")

    for col in df_c.columns:
        print(f"{col}: {df_c[col].unique().tolist()}")

show_categories(df_str)

features: ['convenio', 'pulsos', 'ppa', 'patologia', 'b2', 'sopro', 'fc', 'hda1', 'hda2', 'sexo', 'motivo1', 'motivo2']

convenio: ['GS', 'SULA', nan, 'SAME', 'UR', 'Grupo', 'PART', 'SB', 'SA', 'S.America', 'UG', 'REAL S.', 'Medial', 'SEMEPE', 'CT', 'CC', 'GRUPO', 'S.A', 'Norclínicas', 'SUS', 'SulAmerica', 'Ideal', 'IDEAL', 'Sta Helena', 'Fusex', 'Real S.', '35971', 'Aeronáutica', 'Tabela', 'Viva', 'América', 'NORCL', 'CAMED', 'U.Recife', 'Camed', 'ST.HELENA', 'BLSERV', 'Aero', 'Medserv', 'NORCLIN.', 'AERON.', 'Fachesf', 'FUSEX', 'Norclinicas', 'BLUELI', 'MMS', 'MEDSERV', 'NORC', 'Norclínica', 'AS', 'SBPREF', 'Unimed Recife', 'Semepe', 'Circulo', 'U G', 'Notredame', 'América S.', 'Real', 'FUNCEF', 'CELPOS', 'Norclinica', 'S. RECIFE', 'FACHESF', 'MEDIAL', 'Sta. Clara', 'Sul Am', 'S. CX', 'ASSEF', 'Fisco', 'OAB Saúde', 'Cassi', 'Assefaz', 'Bluelife', 'Notre Dame', 'Funcef', 'SANT H', 'Petrob', 'ASSEFAZ', 'CAMPE', 'Particular', 'America', 'Sta Clara', 'Unimed', 'Gs', 'Sul America', 'ameri

Como podemos notar, a única feature que não parece categórica é a feature `"fc"` que corresponde à frequência cardíaca, pois ela possui valores numéricos e alguns valores no padrão `%d-%d`, portanto vamos tratar dele posteriormente. Por hora vamos tratar das outras features. Para isso vamos estabelecer uma convenção em relação às features categóricas: elas serão do tipo `category` do `Pandas`, terão strings com letras em minúsculo, sem acentos e separados por underscore (`_`) no lugar do espaço. Para seguir essa convenção vamos aplicar a função `normalize_strings` nessas features.

In [27]:
def normalize_strings(x):
    return (x
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.lower()
        .str.strip()
        .str.split(" ")
        .str.join("_")
        .str.replace("-", "_")
    )

cols = df_str.columns.tolist()
df[cols] = df[cols].apply(normalize_strings)

show_categories(df[cols])

features: ['convenio', 'pulsos', 'ppa', 'patologia', 'b2', 'sopro', 'fc', 'hda1', 'hda2', 'sexo', 'motivo1', 'motivo2']

convenio: ['gs', 'sula', nan, 'same', 'ur', 'grupo', 'part', 'sb', 'sa', 's.america', 'ug', 'real_s.', 'medial', 'semepe', 'ct', 'cc', 's.a', 'norclinicas', 'sus', 'sulamerica', 'ideal', 'sta_helena', 'fusex', '35971', 'aeronautica', 'tabela', 'viva', 'america', 'norcl', 'camed', 'u.recife', 'st.helena', 'blserv', 'aero', 'medserv', 'norclin.', 'aeron.', 'fachesf', 'blueli', 'mms', 'norc', 'norclinica', 'as', 'sbpref', 'unimed_recife', 'circulo', 'u_g', 'notredame', 'america_s.', 'real', 'funcef', 'celpos', 's._recife', 'sta._clara', 'sul_am', 's._cx', 'assef', 'fisco', 'oab_saude', 'cassi', 'assefaz', 'bluelife', 'notre_dame', 'sant_h', 'petrob', 'campe', 'particular', 'sta_clara', 'unimed', 'sul_america', 'part.', 'saude_recife', 'mediservice', 'chesf', 'fachesp', 'faches', 'particl', 'agf', 'mediserv', 'medservice', 'norcli', 'sulam', 'mediser', 'saude_bradesco', 

Como podemos ver, muitas features já estão com valores normalizados e sem redundâncias, então vamos tratar as que ainda não estão totalmente tratadas.

### Feature `"ppa"`

Vamos analisar a feature `"ppa"`. Primeiro vamos olhar para algumas estatísticas e contagens dessa feature, pois ela possui um valor `"#value!"` que indica fortemente um erro do Excel.

In [28]:
df[["ppa"]].describe()

,ppa
count,12656
unique,9
top,nao_calculado
freq,9081


In [29]:
100*df["ppa"].value_counts(normalize=True, dropna=False)

ppa
nao_calculado          70.542997
normal                 13.905073
#value!                11.621223
NaN                     1.685699
has_2_pas               0.567078
pre_hipertensao_pad     0.567078
pre_hipertensao_pas     0.372873
has_1_pas               0.372873
has_1_pad               0.201973
has_2_pad               0.163132
Name: proportion, dtype: float64

Como a ocorrência de `"#value!"` ocupa $10\%$ da nossa feature, vamos converter ela para `nan`.

In [30]:
df["ppa"] = df["ppa"].replace("#value!", np.nan)
100*df["ppa"].value_counts(normalize=True, dropna=False)

ppa
nao_calculado          70.542997
normal                 13.905073
NaN                    13.306921
has_2_pas               0.567078
pre_hipertensao_pad     0.567078
pre_hipertensao_pas     0.372873
has_1_pas               0.372873
has_1_pad               0.201973
has_2_pad               0.163132
Name: proportion, dtype: float64

### Feature `"b2"`

Nessa feature vamos apenas renomear uma categoria, para facilitar o entendimento.

In [31]:
df["b2"] = df["b2"].replace("desdob_fixo", "desdobramento_fixo")
df["b2"].value_counts(dropna=False)

b2
normal                11098
NaN                    1178
hiperfonetica           295
desdobramento_fixo      149
unica                    77
outro                    76
Name: count, dtype: int64

### Feature `"sexo"`

Como podemos ver, nessa feature temos classes redundantes, então vamos unificar essas classes.

In [32]:
df["sexo"].value_counts(dropna=False)

sexo
m                6065
f                4477
indeterminado    1417
masculino         663
feminino          247
NaN                 4
Name: count, dtype: int64

In [33]:
df["sexo"] = (df["sexo"]
    .replace("f", "feminino")
    .replace("m", "masculino")
)

df["sexo"].value_counts(dropna=False)

sexo
masculino        6728
feminino         4724
indeterminado    1417
NaN                 4
Name: count, dtype: int64

### Features `"motivo1"` e `"motivo2"`

Olhando para essas duas features notamos que elas têm um prefixo que segue o padrão `"%d___"`, por isso vamos remover esse prefixo dessas features.

In [34]:
print(df["motivo1"].value_counts(dropna=False), end="\n\n")
print(df["motivo2"].value_counts(dropna=False))

motivo1
5___parecer_cardiologico           5352
6___suspeita_de_cardiopatia        4285
1___cardiopatia_ja_estabelecida    1158
NaN                                1055
2___check_up                        701
7___outro                           322
Name: count, dtype: int64

motivo2
NaN                                    3496
5___cirurgia                           2940
6___sopro                              2349
1___cardiopatia_congenica              1041
outro                                   824
5___atividade_fisica                    652
6___palpitacao/taquicardia/arritmia     415
6___dor_precordial                      392
6___has/dislipidemia/obesidade          240
6___dispneia                            226
6___cianose                             145
1___cardiopatia_adquirida                92
6___cardiopatia_na_familia               35
5___uso_de_cisaprida                      9
6___cansaco                               7
6___alteracoes_de_pulso/perfusao          5
6___cianose_e

In [35]:
df["motivo2"] = df["motivo2"].replace("outro", "1___outro")

def remove_prefix(x):
    return x.str.replace(r"^\d___", "", regex=True)

df[["motivo1", "motivo2"]] = df[["motivo1", "motivo2"]].apply(remove_prefix)

print(df["motivo1"].value_counts(dropna=False), end="\n\n")
print(df["motivo2"].value_counts(dropna=False))

motivo1
parecer_cardiologico           5352
suspeita_de_cardiopatia        4285
cardiopatia_ja_estabelecida    1158
NaN                            1055
check_up                        701
outro                           322
Name: count, dtype: int64

motivo2
NaN                                3496
cirurgia                           2940
sopro                              2349
cardiopatia_congenica              1041
outro                               824
atividade_fisica                    652
palpitacao/taquicardia/arritmia     415
dor_precordial                      392
has/dislipidemia/obesidade          240
dispneia                            226
cianose                             145
cardiopatia_adquirida                92
cardiopatia_na_familia               35
uso_de_cisaprida                      9
cansaco                               7
alteracoes_de_pulso/perfusao          5
cianose_e_dispneia                    5
Name: count, dtype: int64


### Feature `"patologia"`

Por fim, vamos olhar para o target, como podemos notar ela possui dois valores equivalentes.

In [36]:
df["patologia"].value_counts(dropna=False)

patologia
normal     6744
anormal    4960
NaN        1168
normais       1
Name: count, dtype: int64

Vamos transformar os valores `"normais"` em valores `"normal"`. Além disso, vamos converter essa feature para o tipo booleano, onde `True` significa presença de patologia cardíaca e `False` a ausência, para ser semanticamente mais correto.

In [37]:
df["patologia"] = df["patologia"].replace("normais", "normal")
df["patologia"] = df["patologia"].map({"normal": False, "anormal": True}).astype("boolean")

In [38]:
df["patologia"].value_counts(dropna=False)

patologia
False    6745
True     4960
<NA>     1168
Name: count, dtype: Int64

In [39]:
df_dt = df.convert_dtypes()

df_dt.info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             12873 non-null  Int64         
 1   peso           9945 non-null   Float64       
 2   altura         8413 non-null   Int64         
 3   imc            8037 non-null   Int64         
 4   atendimento    11577 non-null  datetime64[us]
 5   nascimento     11242 non-null  datetime64[us]
 6   idade          11255 non-null  Float64       
 7   convenio       8765 non-null   string        
 8   pulsos         11679 non-null  string        
 9   pa_sistolica   5143 non-null   Int64         
 10  pa_diastolica  5133 non-null   Int64         
 11  ppa            11160 non-null  string        
 12  patologia      11705 non-null  boolean       
 13  b2             11695 non-null  string        
 14  sopro          11706 non-null  string        
 15  fc             11005 non-null 

### Feature `"fc"`

Por fim, vamos analisar a feature `"fc"`. Vamos olhar algumas estatísticas e contagens dela.

In [40]:
df[["fc"]].describe()

,fc
count,11005
unique,118
top,100
freq,1509


In [41]:
print(df["fc"].value_counts(dropna=False).to_string())

fc
NaN          1868
100          1509
80           1480
90           1104
88            832
96            782
120           780
92            599
72            456
110           437
78            359
102           214
84            214
112           181
82            171
130           171
70            165
140           157
76            149
94            100
98             94
150            80
128            70
68             68
95             67
132            60
75             59
108            57
85             56
104            56
60             45
160            45
86             33
136            25
64             23
124            22
74             21
135            17
116            16
148            12
115            11
65             11
138            10
144             8
66              8
125             8
180             7
142             7
89              7
134             6
50              6
56              6
122             6
118             6
156             5
106    

Como podemos ver, existem pouquíssimos registros com valores de frequência cardíaca no padrão `%d_%d`, então vamos converter esses valores para a média dos dois inteiros na string.

In [42]:
import re

def convert_fc(x):
    if pd.isna(x):
        return x
    
    match = re.match(r"^(\d+)[_a]+(\d+)$", x)

    if match:
        a = int(match[1])
        b = int(match[2])

        return (a + b) // 2
    
    return int(x)

df["fc"] = df["fc"].apply(convert_fc)

In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             12873 non-null  int64         
 1   peso           9945 non-null   float64       
 2   altura         8413 non-null   float64       
 3   imc            8037 non-null   float64       
 4   atendimento    11577 non-null  datetime64[us]
 5   nascimento     11242 non-null  datetime64[us]
 6   idade          11255 non-null  float64       
 7   convenio       8765 non-null   object        
 8   pulsos         11679 non-null  object        
 9   pa_sistolica   5143 non-null   float64       
 10  pa_diastolica  5133 non-null   float64       
 11  ppa            11160 non-null  object        
 12  patologia      11705 non-null  boolean       
 13  b2             11695 non-null  object        
 14  sopro          11706 non-null  object        
 15  fc             11005 non-null 

### Criando as Features Categóricas

Vamos converter as features do tipo `object` para o tipo `category` do `Pandas`.

In [44]:
cols = df.select_dtypes(include="object").columns

df[cols] = df[cols].astype("category")
df[cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   convenio  8765 non-null   category
 1   pulsos    11679 non-null  category
 2   ppa       11160 non-null  category
 3   b2        11695 non-null  category
 4   sopro     11706 non-null  category
 5   hda1      8603 non-null   category
 6   hda2      407 non-null    category
 7   sexo      12869 non-null  category
 8   motivo1   11818 non-null  category
 9   motivo2   9377 non-null   category
dtypes: category(10)
memory usage: 143.7 KB


## Salvando o Dataset

Por fim, vamos converter os tipos do dataset para os tipos mais apropriados no Pandas e vamos salvar ele em formato parquet.

In [45]:
df = df.convert_dtypes()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12873 entries, 0 to 12872
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             12873 non-null  Int64         
 1   peso           9945 non-null   Float64       
 2   altura         8413 non-null   Int64         
 3   imc            8037 non-null   Int64         
 4   atendimento    11577 non-null  datetime64[us]
 5   nascimento     11242 non-null  datetime64[us]
 6   idade          11255 non-null  Float64       
 7   convenio       8765 non-null   category      
 8   pulsos         11679 non-null  category      
 9   pa_sistolica   5143 non-null   Int64         
 10  pa_diastolica  5133 non-null   Int64         
 11  ppa            11160 non-null  category      
 12  patologia      11705 non-null  boolean       
 13  b2             11695 non-null  category      
 14  sopro          11706 non-null  category      
 15  fc             11005 non-null 

In [46]:
df.to_parquet("../data/processed/UCMF_processed.parquet")